### **Table 4: Baseline Comparison**  

## Objective  

In this experiment, we compare the performance of our proposed baseline method against the original authors' baseline across different game variants. The goal is to assess the effectiveness of our method in maintaining consistency across games while serving as a reference point for evaluating negotiation models. Since we were unable to reproduce the original authors' baseline results, this notebook only evaluates our proposed method. To create **Table 4** in our paper, we include the original baseline metrics as reported in their paper. This experiment is referred to in our paper as **Experiment 5**.

---

## Methodology  

- **Baseline Method Evaluated:**  
  - Our proposed baseline method  

- **Evaluation Metrics:**  
  The script computes the following metrics for our baseline method across different games:  
  - % 5/6-way agreement  
  - % 6-way agreement  

---

## Results  
The results of this experiment validate the effectiveness and consistency of our proposed baseline method across multiple negotiation games. These findings are presented in **Table 4** of our paper.

In [ ]:
import os
import eval_utils as evaluation
import json
import pandas as pd
from IPython.display import display

method = 'repeated_rule_based'

games = ['base', 'game1', 'game2', 'game3']
results = {}
ISSUES_NUM = 5
AGENTS_NUM = 6

for game in games:
    directory = f'../our_games_descriptions/{game}/output/baselines/{method}'

    agents, role_to_agents, incentive_to_agents = evaluation.load_setup(directory, AGENTS_NUM, num_issues=ISSUES_NUM)
    answers_files = [ os.path.join(directory,filename) for filename in os.listdir(directory) if filename.startswith("history")]

    num_rounds = 0
    for file_ in answers_files:
        answers = json.load(open(file_))
        _num_rounds = len(answers['rounds'])
        num_rounds = max(num_rounds, _num_rounds)

    # Track statistics
    feasible_in_last_step = 0
    accepted_by_all_in_last_step = 0
    contained_feasible_deal = 0
    successfull_games = 0
    total_rounds = 0

    # Loop through all answer files (each represents a game)
    for file_ in answers_files:
        answers = json.load(open(file_))
        
        if len(answers['rounds']) != num_rounds:
            print(f"WARNING: Game {file_} has a different number of rounds")
            continue
        total_rounds += len(answers['rounds'])
        successfull_games += 1

        # Extract deals for this game
        feasible_found = False

        # Extract the name of the first player (p1) to validate feasibility throughout the game
        p1_name = answers['rounds'][0]['agent']

        total_deals = 0
        
        for i, round_ in enumerate(answers['rounds']):
            name, answer = round_['agent'], round_['public_answer']
            deal_unformatted, issues_suggested = evaluation.extract_deal(answer, ISSUES_NUM)

            try:
                deal = evaluation.format_deal(deal_unformatted, ISSUES_NUM)
            except:
                print(f"Error in game {file_} round {i}")
                continue

            if issues_suggested >= ISSUES_NUM:
                total_deals += 1

            # Check if the deal was feasible at any point (Deal must have been proposed by p1)
            if evaluation.is_feasible(agents, deal) and name == p1_name:
                feasible_found = True
        

        # CHECK GAME COMPLETION METRICS

        last_deal = evaluation.format_deal(evaluation.extract_deal(answers['rounds'][-1]['public_answer'], ISSUES_NUM)[0], ISSUES_NUM)
        
        # 1. Check if the last deal is feasible
        if evaluation.is_feasible(agents, last_deal):
            feasible_in_last_step += 1

        # 2. Check if the last deal is acceptable by all agents
        all_accept = all(evaluation.calculator(agents[agent]["scores"], last_deal, ISSUES_NUM, verbose=False) >= agents[agent]["scores"]["min"] for agent in agents)
        if all_accept:
            accepted_by_all_in_last_step += 1

        # 3. Check if any deal during the game was in the feasibility set
        if feasible_found:
            contained_feasible_deal += 1

    # Compute percentages
    num_games = successfull_games
    perc_feasible_last = (feasible_in_last_step / num_games) * 100
    perc_accepted_all_last = (accepted_by_all_in_last_step / num_games) * 100
    perc_feasible_any = (contained_feasible_deal / num_games) * 100

    results[game] = {
        '5/6-way (%)': perc_feasible_last,
        '6-way (%)': perc_accepted_all_last,
        'Any (%)': perc_feasible_any,
    }

df = pd.DataFrame(results).T
display(df)

,5/6-way (%),6-way (%),Any (%)
base,62.6,47.0,85.6
game1,79.1,69.8,88.0
game2,68.8,57.8,83.7
game3,82.5,81.7,85.0
